In [1]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from glob import glob
import requests
import re
import os,shutil,sys,traceback
import pprint
import subprocess

import astropy
from astropy.table import Table
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time
from astropy.wcs import WCS

from astroquery.mast import Observations

from photutils.detection import DAOStarFinder, StarFinder
from photutils.aperture import RectangularAperture, RectangularAnnulus,CircularAperture, CircularAnnulus
from photutils.aperture import ApertureStats, aperture_photometry

# import space_phot

%matplotlib widget

In [2]:
# This is a utility function to allow us to visualize things in DS9
def create_pixregionfile(x,y,regionname,color,coords='image',radius=1):
    if isinstance(radius,int):
        radius = [radius]*len(x)
    with open(regionname, 'w') as f:
        if isinstance(color,str):
            f.write('global color={0} dashlist=8 3 width=2 font=\"helvetica 10 normal roman\" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1 \n'.format(color))
            do_col = False
        else:
            do_col = True
            f.write('global dashlist=8 3 width=2 font=\"helvetica 10 normal roman\" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1 \n')
        f.write('%s \n'%coords)
        for star in range(len(x)):
            xval = x[star]
            yval = y[star]
            if do_col:
                f.write('circle({ra},{dec},{radius}") # color={color}\n'.format(ra=xval, dec=yval,radius=radius[star],color=color[star]))
            else:
                f.write('circle({ra},{dec},{radius}")\n'.format(ra=xval, dec=yval,radius=radius[star]))
    f.close()

In [3]:
# # Function wrapper for JHAT
# def align_and_create_region_files(input_file, output_subdir, reference_cat_name, max_depth):
    
#     wcs_align = st_wcs_align()
    
#     try:
#         print("\t***** Processing File: `%s` *****" % f)
    
#         wcs_align.run_all(input_file,
#                       telescope='jwst',
#                       outsubdir=output_subdir,
#                       refcat_racol='RA',
#                       refcat_deccol='DEC',
#                       refcat_magcol='MAG',
#                       refcat_magerrcol='MAG_ERR',
#                       overwrite=True,
#                       d2d_max=0.25,
#                       showplots=0,
#                       refcatname=reference_cat_name,
#                       histocut_order='dxdy',
#                       sharpness_lim=(0.3,0.9), # 0 to 1
#                       roundness1_lim=(-0.7, 0.7), # -1 to 1
#                       SNR_min= 3,
#                       dmag_max=1.0,
#                       objmag_lim =(14,max_depth),
#                       saveplots=2,
#                       savephottable=True)
    
#         good_matches_file_name = os.path.basename(f).replace('_cal.fits', '.goodmatches.csv')
#         good_matches_file_path = "./{base_path}/{file_name}".format(base_path=jhat_epoch1_outsubdir, file_name=good_matches_file_name)
#         good_match_tbl = Table.read(good_matches_file_path, format="ascii")

#         create_pixregionfile(good_match_tbl["RA"], 
#                              good_match_tbl["DEC"],
#                              good_matches_file_path.replace("csv", "reg"), 
#                              color="red", coords="icrs", radius=[0.4]*len(good_match_tbl))
        
              
#     except Exception as e:
#         print("\t\t***** FAILED Processing File: `%s` *****" % f)
#         print(e)

In [48]:
# This is a function that wraps a HOTPANTS call.
def run_hotpants(ref_input_file, sci_input_file, curr_working_dir, diff_dir, hotpants_params):

    # Utility function to take multiextension FITS files and save out each extension as its own file. 
    # HOTPANTS requires single extension files.

    def unpack_fits(input_file, sci_out_filename, err_out_filename, stamp_outfile, ref = True):
        
        # --- Unpack data into single extension files ---
        with fits.open(input_file, output_verify='fix') as dat:

            sci_data = dat['IMAGE'].data
            max_sci_val = np.nanmax(sci_data)
            sci_data[~np.isfinite(sci_data)] = 0
        
            err_data = dat['UNCERTAINTY'].data
            err_data[~np.isfinite(err_data)] = 0
            
            sci_pedastal = 500
            dat[0].data = sci_data + sci_pedastal
            dat[0].header = dat[0].header + dat[1].header
            dat[0].writeto(sci_out_filename, overwrite=True)
        
            dat[0].data = err_data
            dat[0].writeto(err_out_filename, overwrite=True)

        # --- Run astrometry.net solve-field on sci_out_filename ---
        astrom_call = f"solve-field --overwrite --no-plots --scale-units arcsecperpix --scale-low 3 --scale-high 5 -p {sci_out_filename}"
        
        with subprocess.Popen(astrom_call, stdout=subprocess.PIPE, shell=True) as proc:
            stdout, _ = proc.communicate()

        # --- Find the new WCS FITS file generated by solve-field ---
        wcs_file = sci_out_filename.replace(".fits", ".new")  # astrometry.net creates <filename>.new
        
        if os.path.exists(wcs_file):
            # Replace sci_out_filename with the WCS-updated file
            os.replace(wcs_file, sci_out_filename)
        else:
            print("WARNING: WCS solution not found, keeping original sci_out_filename")

        # --- Clean up extra astrometry.net files ---
        for ext in ["*.xyls", "*.axy", "*.match", "*.rdls", "*.corr", "*.wcs", "*.solved"]:
            for f in glob(sci_out_filename.replace(".fits", ext)):
                os.remove(f)
        
        phot_table = None 
        if ref:
            new_hdu = fits.open(sci_out_filename)
            wcs = WCS(new_hdu[0].header)
            wcs_data = new_hdu[0].data
            new_hdu.close()
            
            # --- Star finding + photometry ---
            daofind = DAOStarFinder(threshold=550, fwhm=1.5)
            sources = daofind(wcs_data)
            
            if sources is not None:
                sources = sources[(sources['sharpness'] > 0.4) & (abs(sources['roundness1']) < 0.9)]
                sources = sources.to_pandas()
                positions = np.column_stack([sources['xcentroid'], sources['ycentroid']])
                
                ras, decs = wcs.all_pix2world(positions[:,0], positions[:,1], 0)
                
                aperture = CircularAperture(positions, 2.7)
                phot_table = aperture_photometry(wcs_data, aperture)
                phot_table = phot_table.to_pandas()
                phot_table = phot_table.drop(columns=['id']).sort_values(by=['aperture_sum'], ascending=False)
                phot_table = phot_table.reset_index(drop=True)
                
                flux = phot_table['aperture_sum'].values
                output_tbl = Table([positions[:,0] + 1, positions[:,1] + 1, ras, decs, flux], names=["X", "Y", "RA", "DEC", "FLUX"])
                output_tbl.write(stamp_outfile, format="ascii", overwrite=True)
                
                # np.savetxt(stamp_outfile, np.column_stack([positions[:,0]+1, positions[:,1]+1]), fmt="%.2f %.2f")
                
                create_pixregionfile(positions[:,0], positions[:,1], stamp_outfile.replace("txt", "reg"), color="red",
                                    coords="image", radius=[0.4] * len(positions[:,1]))
                
            else:
                sources = []
        
        return max_sci_val
    
    # Utility function to mask out regions with NaNs
    def create_mask(input_err_file, output_mask_file):

        with fits.open(input_err_file, output_verify='fix') as dat:
            sci_mask = np.zeros_like(dat[0].data)
            sci_mask[dat[0].data == 0] = 0x80
            sci_mask[np.isnan(dat[0].data)] = 0x80
        
            dat[0].data = sci_mask
            dat[0].scale('int16')
            dat.writeto(output_mask_file, overwrite=True)

    # # Takes Tabular Fits file w/ specific columns... feel free to generalize
    # def create_stamp_catalog(reference_cat_fits, input_file, filt_name, flux_thresh, stamp_outfile):
        
    #     # Build stampxy file
    #     with fits.open(reference_cat_fits, memmap=True) as cat_file:
        
    #         table = Table(cat_file["CIRC_BSUB"].data)            
            
    #         flux_key = "{filt}_CIRC0".format(filt=filt_name)
    #         err_key = "{filt}_CIRC0_e".format(filt=filt_name)
            
    #         ras = table["RA"]
    #         decs = table["DEC"]
        
    #         flux = table[flux_key]
    #         flux_err = table[err_key]

    #         # reverse sort so brightest sources first
    #         indices = (-flux).argsort()
        
    #         sorted_ra = ras[indices]
    #         sorted_dec = decs[indices]
    #         sorted_fluxes = flux[indices]
    #         sorted_flux_err = flux_err[indices]

    #         # get sources brighter than some threshold
    #         bright_ind = np.where(sorted_fluxes > flux_thresh)[0]
    #         bright_ra = sorted_ra[bright_ind]
    #         bright_dec = sorted_dec[bright_ind]
    #         bright_flux = sorted_fluxes[bright_ind]
    #         bright_flux_err = sorted_flux_err[bright_ind]
                 
    #         coords = SkyCoord(bright_ra, bright_dec, unit=(u.deg, u.deg))
    #         sci_obs3 = space_phot.observation3(input_file)
    #         y_max, x_max = sci_obs3.data.shape
    #         xs, ys = sci_obs3.wcs.world_to_pixel(coords)
    #         in_img_indices = np.where((xs >= 0) & (xs <= x_max) & (ys >= 0) & (ys <= y_max))[0]
            
    #         output_tbl = Table(
    #             [xs[in_img_indices],
    #              ys[in_img_indices],
    #              bright_ra[in_img_indices],
    #              bright_dec[in_img_indices],
    #              bright_flux[in_img_indices]], names=["X", "Y", "RA", "DEC", "FLUX"])
        
    #         output_tbl.write(stamp_outfile, format="ascii", overwrite=True)
    #         create_pixregionfile(xs[in_img_indices], ys[in_img_indices], stamp_outfile.replace("txt", "reg"), color="red",
    #                              coords="image", radius=[0.4] * len(ys[in_img_indices]))

    
    # Create single extension file names for Science and Reference images
    
    sci_output_file = os.path.join(curr_working_dir, "Test_Data", sci_input_file.replace(".fits", "_1.fits").split('/')[-1])
    sci_output_err_file = os.path.join(curr_working_dir, "Test_Data", sci_input_file.replace(".fits", "_1.noise.fits").split('/')[-1])
    sci_output_mask_file = os.path.join(curr_working_dir, "Test_Data", sci_input_file.replace(".fits", "_1.mask.fits").split('/')[-1])
    
    # sci_output_err_file = sci_input_file.replace(".fits", "_1.noise.fits")
    # sci_output_mask_file = sci_input_file.replace(".fits", "_1.mask.fits")
    # sci_stampxy_file = sci_input_file.replace('.fits', '.stampxy.txt')
    sci_stampxy_file = os.path.join(curr_working_dir, "Test_Data", sci_input_file.replace(".fits", ".stampxy.txt").split('/')[-1])
    max_val_sci_im = unpack_fits(sci_input_file, sci_output_file, sci_output_err_file, sci_stampxy_file, ref = False)

    # ref_output_file = ref_input_file.replace(".fits", "_1.fits")
    # ref_output_err_file = ref_input_file.replace(".fits", "_1.noise.fits")
    # ref_output_mask_file = ref_input_file.replace(".fits", "_1.mask.fits")
    ref_output_file = os.path.join(curr_working_dir, "Test_Data", ref_input_file.replace(".fits", "_1.fits").split('/')[-1])
    ref_output_err_file = os.path.join(curr_working_dir, "Test_Data", ref_input_file.replace(".fits", "_1.noise.fits").split('/')[-1])
    ref_output_mask_file = os.path.join(curr_working_dir, "Test_Data", ref_input_file.replace(".fits", "_1.mask.fits").split('/')[-1])
    max_val_ref_im = unpack_fits(ref_input_file, ref_output_file, ref_output_err_file, sci_stampxy_file, ref = True)
    
    print('STAMP:', sci_stampxy_file)

    # Create masks for Science and Reference images
    create_mask(sci_output_err_file, sci_output_mask_file)
    create_mask(ref_output_err_file, ref_output_mask_file)
    
    # Dymanically create the Difference Image file name
    diff_file_basename_formatter = "testing.diff.fits"

    # Set up the named files to capture HOTPANTS output
    diff_file = os.path.join(diff_dir, 
                             diff_file_basename_formatter)#.format(filt_name=filt_name, sci_epoch=sci_file_epoch, ref_epoch=ref_file_epoch))
    diff_err_file = diff_file.replace('.fits', '.noise.fits')
    diff_mask_file = diff_file.replace('.fits', '.mask.fits')
    diff_kernel_file = diff_file.replace('.fits', '.kernel.fits') 
    diff_stampxy_reg_file = diff_file.replace('.fits', '.stampxy.reg')
    
    def to_docker_path(path):
        new_path = path.replace(curr_working_dir + '/', "")
        return new_path

    sci_output_file_docker = to_docker_path(sci_output_file)
    ref_output_file_docker = to_docker_path(ref_output_file)
    sci_output_err_file_docker = to_docker_path(sci_output_err_file)
    ref_output_err_file_docker = to_docker_path(ref_output_err_file)
    sci_output_mask_file_docker = to_docker_path(sci_output_mask_file)
    ref_output_mask_file_docker = to_docker_path(ref_output_mask_file)
    sci_stampxy_file_docker = to_docker_path(sci_stampxy_file)
    diff_file_docker = to_docker_path(diff_file)
    diff_err_file_docker = to_docker_path(diff_err_file)
    diff_mask_file_docker = to_docker_path(diff_mask_file)
    diff_kernel_file_docker = to_docker_path(diff_kernel_file)
    diff_stampxy_reg_file_docker = to_docker_path(diff_stampxy_reg_file)
    
    print(sci_output_file_docker, ref_output_file_docker, diff_file)

    # Build the catalog for HOTPANTS to sample the PSFs
    # create_stamp_catalog(cat_file, sci_output_file, filt_name, flux_thresh=10.0, stamp_outfile=sci_stampxy_file)

    # A bit dodgey: We're assuming the same instrument and filter between science and template image
    sci_fwhm = 1.35

    # Update dynamic parameters in the config dictionary
    hotpants_params['iu'] = max_val_sci_im
    hotpants_params['iuk'] = max_val_sci_im
    
    hotpants_params['tu'] = max_val_ref_im
    hotpants_params['tuk'] = max_val_ref_im
    
    hotpants_params['r'] = 2.5 * sci_fwhm
    hotpants_params['rss'] = 2.5 * sci_fwhm

    # ngauss degree0 sigma0 .. degreeN sigmaN]
    # : ngauss = number of gaussians which compose kernel (3)
    # : degree = degree of polynomial associated with gaussian #
    #            (6 4 2)
    # : sigma  = width of gaussian #
    #            (0.70 1.50 3.00)
    # : N = 0 .. ngauss - 1
    hotpants_params['ng'] = (3, 6, sci_fwhm / 2.0, 4, sci_fwhm, 2, sci_fwhm * 2.0)

    # Construct the hotpants command string

    # Boolean flags:
    #    -savexy # set file name . saves the X,Y positions of used, clipped, and all substamps in different colors
    #    -sconv # all regions convolved in same direction (0)
    
    hotpants_arg = (
        f'hotpants -inim {sci_output_file_docker.strip()} -tmplim {ref_output_file_docker.strip()} -outim {diff_file.strip()} '
        f'-ini {sci_output_err_file_docker.strip()} ' #-imi {sci_output_mask_file_docker.strip()} 
        f'-il {hotpants_params["il"]} '
        f'-iu {hotpants_params["iu"]} -iuk {hotpants_params["iuk"]} -tni {ref_output_err_file_docker} '
        # f'-tmi {ref_output_mask_file_docker} 
        f'-tl {hotpants_params["tl"]} -tu {hotpants_params["tu"]} '
        f'-tuk {hotpants_params["tuk"]} -nrx {hotpants_params["nrx"]} -nry {hotpants_params["nry"]} '
        f'-nsx {hotpants_params["nsx"]} -nsy {hotpants_params["nsy"]} -nss {hotpants_params["nss"]} '
        f'-ng {" ".join(map(str, hotpants_params["ng"]))} -rss {hotpants_params["rss"]} -ft {hotpants_params["ft"]} '
        f'-r {hotpants_params["r"]} -ko {hotpants_params["ko"]} -bgo {hotpants_params["bgo"]} '
        f'-ssig {hotpants_params["ssig"]} -ks {hotpants_params["ks"]} -kfm {hotpants_params["kfm"]} -okn '
        f'-c {hotpants_params["c"]} -n {hotpants_params["n"]} -sconv -cmp {sci_stampxy_file_docker} '
        f'-afssc {hotpants_params["afssc"]} -gridssc {hotpants_params["gridssc"]} -fi {hotpants_params["fi"]} '
        f'-oni {diff_err_file} -fin {hotpants_params["fin"]} -mins {hotpants_params["mins"]} '
        f'-omi {diff_mask_file} -mous {hotpants_params["mous"]} -oki {diff_kernel_file} '
        f'-v {hotpants_params["v"]} -savexy {diff_stampxy_reg_file}'
    )
    
    print("Hotpants invocation:\n\t%s" % hotpants_arg)

    # This is where you need Docker!
    os.system("docker run --rm -v %s:/app ghcr.io/davecoulter/jwst_ss_diffim:1.0.0 %s" % (curr_working_dir, hotpants_arg))

    return diff_file


In [49]:
# We need to map our current working path to the docker container to get output locally
local_working_dir = os.getcwd()

# Let's build our HOTPANTS parameter file!
hotpants_config = {
    'ko': 2, # spatial order of kernel variation within region (2)
    'bgo': 2, # spatial order of background variation within region (1)
    'ssig': 3.0, # threshold for sigma clipping statistics  (3.0)
    'ks': 2.0, # high sigma rejection for bad stamps in kernel fit (2.0)
    'kfm': 0.99, # fraction of abs(kernel) sum for ok pixel (0.990)
    'il': 0, # lower valid data count, image (0)
    'tl': 0, # lower valid data count, template (0)
    'nrx': 1, # number of image regions in x dimension (1)
    'nry': 1, # number of image regions in y dimension (1)
    # 'nsx': 5, # number of each region's stamps in x dimension (10)
    # 'nsy': 5, # number of each region's stamps in y dimension (10)
    'nsx': 8, # number of each region's stamps in x dimension (10)
    'nsy': 8, # number of each region's stamps in y dimension (10)
    # 'nsx': 20, # number of each region's stamps in x dimension (10)
    # 'nsy': 20, # number of each region's stamps in y dimension (10)
    'nss': 7, # number of centroids to use for each stamp (3) # DC: got a segmentation fault using "10"
    'ft': 5.0, # RMS threshold for good centroid in kernel fit (20.0)
    'c': 't', # force convolution on (t)emplate or (i)mage (undef)
    'n': 'i', # normalize to (t)emplate, (i)mage, or (u)nconvolved (t)
    'afssc': 0, # autofind stamp centers so #=-nss when -ssf,-cmp (1)
    'gridssc': 0, 
    'fi': 0.0, # value for invalid (bad) pixels (1.0e-30)
    'fin': 0.0, # noise image only fillvalue (0.0e+00)
    'mins': 2.0, # Fraction of kernel half width to spread input mask (1.0)
    'mous': 0.0, # Ditto output mask, negative = no diffim masking (1.0)
    'v': 2, # level of verbosity, 0-2 (1)
    
    'r': 0, # Will be calculated dynamically
    'rss': 0, # Will be calculated dynamically
    'ng': (), # Will be calculated dynamically
    'iu': 0, # Will be calculated dynamically
    'iuk': 0, # Will be calculated dynamically
    'tu': 0, # Will be calculated dynamically
    'tuk': 0 # Will be calculated dynamically
}

In [50]:
files = glob('/Users/zgl12/Research/SuperstampMe/*.fits')

cads = [int(file.split('cad')[-1].split('.fit')[0]) for file in files]
indices = list(range(len(cads)))

df = pd.DataFrame([], columns = ['cadence', 'files'])

df['cadence'] = cads
df['files'] = files

df = df.sort_values('cadence')
df = df.reset_index(drop = True)

In [51]:
# input_file = 'Test_Data/test_sci.fits'
# tmp = 'Temp'
# # name = 'Test_Data/test_sci_wcs.fits'

# astrom_call = f"solve-field --no-plots --scale-units arcsecperpix --scale-low 3 --scale-high 5 --temp-dir {tmp} -p {input_file}"

# with subprocess.Popen(astrom_call, stdout=subprocess.PIPE, shell=True) as proc:
#                         # Wait for the process to finish and capture its output
#                         stdout, _ = proc.communicate()

In [52]:
# Create output dir
diff_outsubdir = "./diff"
os.makedirs(diff_outsubdir, exist_ok=True)

# f200w_ref = "%s/%s_i2d.fits" % (ep1_output_dir, f200w_epoch1_basefile)
# f200w_sci = "%s/%s_i2d.fits" % (ep2_output_dir, f200w_epoch2_basefile)

# ref_fits = "./JADES_data/hlsp_jades_jwst_nircam_goods-s-deep_photometry_v2.0_catalog.fits"

# f200w_diff_im = run_hotpants(inst_name="NIRCAM", 
#              filt_name="F200W", 
#              ref_input_file=f200w_ref, 
#              sci_input_file=f200w_sci, 
#              cat_file=ref_fits, 
#              curr_working_dir=local_working_dir,
#              diff_dir=diff_outsubdir, 
#              hotpants_params=hotpants_config)


hotpants = run_hotpants(ref_input_file = df['files'][200], 
                        sci_input_file = df['files'][2800], 
                        curr_working_dir = local_working_dir, 
                        diff_dir = diff_outsubdir, 
                        hotpants_params = hotpants_config)

Set DATE-BEG to '2014-11-19T16:47:04.071' from MJD-BEG'. [astropy.wcs.wcs]


STAMP: /Users/zgl12/Modules/Kakapo/HotPants_Test/Test_Data/k2mosaic-c03-ch80-cad102407.stampxy.txt
Test_Data/k2mosaic-c03-ch80-cad102407_1.fits Test_Data/k2mosaic-c03-ch80-cad99800_1.fits ./diff/testing.diff.fits
Hotpants invocation:
	hotpants -inim Test_Data/k2mosaic-c03-ch80-cad102407_1.fits -tmplim Test_Data/k2mosaic-c03-ch80-cad99800_1.fits -outim ./diff/testing.diff.fits -ini Test_Data/k2mosaic-c03-ch80-cad102407_1.noise.fits -il 0 -iu 207045.546875 -iuk 207045.546875 -tni Test_Data/k2mosaic-c03-ch80-cad99800_1.noise.fits -tl 0 -tu 205120.90625 -tuk 205120.90625 -nrx 1 -nry 1 -nsx 8 -nsy 8 -nss 7 -ng 3 6 0.675 4 1.35 2 2.7 -rss 3.375 -ft 5.0 -r 3.375 -ko 2 -bgo 2 -ssig 3.0 -ks 2.0 -kfm 0.99 -okn -c t -n i -sconv -cmp Test_Data/k2mosaic-c03-ch80-cad102407.stampxy.txt -afssc 0 -gridssc 0 -fi 0.0 -oni ./diff/testing.diff.noise.fits -fin 0.0 -mins 2.0 -omi ./diff/testing.diff.mask.fits -mous 0.0 -oki ./diff/testing.diff.kernel.fits -v 2 -savexy ./diff/testing.diff.stampxy.reg


Doing : Test_Data/k2mosaic-c03-ch80-cad102407_1.fits -
        Test_Data/k2mosaic-c03-ch80-cad99800_1.fits =
        ./diff/testing.diff.fits
   Good templ data : 0.0 -> 205120.9
   Good image data : 0.0 -> 207045.5
Mallocing massive amounts of memory...
Region 0 pixels            : 1:1132,1:1070
 Vector Indices (buffered) : 0:1131,0:1069
 Vector Indices (good data): 0:1131,0:1069
Allocating image stamps...
Allocating template stamps...
region size: (X,Y)=(1132,1070)
# stamps: (X,Y)=(8,8)
stamp size: (X,Y)=(141,133)
Build stamp  : t    0 i    0 (grid coord  0  0)

Adding centers manually to s(0,0) x:0-140 y:0-132
X stamp: 0-140 region:0-1131
Y stamp: 0-132 region:0-1069
    Stamp in region : 0:140,0:132 X=24.3 Y=51.3
     #0 @   24,  51
    Stamp in region : 0:140,0:132 X=92.7 Y=71.0
     #1 @   92,  71
    Stamp in region : 0:140,0:132 X=77.7 Y=84.6
     #2 @   77,  84
    templ: 3 substamps
Build stamp  : t    1 i    0 (grid coord  1  0)

Adding centers manually to s(1,0) x:141-281 y

In [47]:
hdu = fits.open('/Users/zgl12/Modules/Kakapo/HotPants_Test/diff/testing.diff.fits')
hdu.info()
hdu.close()

Filename: /Users/zgl12/Modules/Kakapo/HotPants_Test/diff/testing.diff.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU    1002   (1132, 1070)   float32   
